# Case Study 12 — Afghanistan Crop Growth Monitoring

This case study demonstrates the use of the Mean Vegetation Health
Index (MVHI) for crop-growth monitoring, using Afghanistan's first
growing season (GS1) as an example. The latest available MVHI image is
compared with the multiyear mean for the same dekadal phase to assess
current crop conditions relative to the historical baseline.

The workflow searches the FAO STAC catalogue, downloads the required
LC-C rasters, clips them to Afghanistan, calculates pixel-wise
multiyear means for GS1 and GS2 separately, and saves the final GS1
comparison figure. Source GCS data are read only; all outputs are saved
locally.


In [1]:
# ============================================================
# 1. IMPORT PACKAGES
# ============================================================

import gc
import os
import re
import shutil
import tempfile
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urlparse

import fiona
import gcsfs
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import Normalize
from pystac_client import Client
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.warp import reproject, transform_geom


Cannot find header.dxf (GDAL_DATA is not defined)


In [11]:
# ============================================================
# 2. SETTINGS
# ============================================================

API_ROOT = "https://data.apps.fao.org/geospatial/search/stac"
COLLECTION_ID = "fao-gismgr:ASIS:raster:mapsets:MVHI-D"

# Afghanistan bounding box in WGS84.
BBOX = [60, 29, 75, 39]

# The 25-day lag avoids requesting data that may not yet be published.
START_DATE = "1992-01-01"
END_DATE = (
    datetime.now() - timedelta(days=25)
).strftime("%Y-%m-%d")
DATETIME_RANGE = f"{START_DATE}/{END_DATE}"

# SQL LIKE pattern used by the STAC CQL2 filter.
ITEM_ID_LIKE = "%LC-C"

# STAC page size. search.items() follows all pages automatically.
PAGE_LIMIT = 100

# ------------------------------------------------------------
# Local paths — current Windows test environment
# ------------------------------------------------------------
BASE_DIR = Path(r"F:\NotebookTest\Case12\AverageASIS-test")
BOUNDARY_FILE = Path(r"D:\GeoBasicData\GlobalAdmin\Global.shp")

# VM alternatives:
# BASE_DIR = Path("/RSP_test/AverageASIS-test")
# BOUNDARY_FILE = Path(
#     "/home/pengyu_hao_fao_org/Boundary/Global/shp/Global.shp"
# )

TEMP_ROOT = BASE_DIR / "tmp"
OUTPUT_DIR = BASE_DIR / "output"

TEMP_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# None = calculate every phase found in the STAC results.
# Test example: ["06-D1"]
PHASES_TO_RUN = None
SEASONS_TO_RUN = ["GS1", "GS2"]

# Existing mean GeoTIFFs are skipped when False.
OVERWRITE_MEANS = False

# Do not save an incomplete mean if any annual raster fails.
STRICT_ALL_YEARS = True

VALID_MIN = 0.0
VALID_MAX = 1.0
OUTPUT_NODATA = 255.0
GCS_TOKEN = "google_default"
#GCS_TOKEN = "browser"
# Latest-observation comparison settings.
LATEST_SEASON = "GS1"
MEAN_FILE_OVERRIDE = None
OVERWRITE_FIGURE = True
FIGURE_DPI = 300

print("STAC API:", API_ROOT)
print("Collection:", COLLECTION_ID)
print("Date range:", DATETIME_RANGE)
print("Temporary directory:", TEMP_ROOT)
print("Output directory:", OUTPUT_DIR)
print(GCS_TOKEN)

STAC API: https://data.apps.fao.org/geospatial/search/stac
Collection: fao-gismgr:ASIS:raster:mapsets:MVHI-D
Date range: 1992-01-01/2026-09-12
Temporary directory: F:\NotebookTest\Case12\AverageASIS-test\tmp
Output directory: F:\NotebookTest\Case12\AverageASIS-test\output
google_default


In [3]:
# ============================================================
# 3. SEARCH MVHI-D LC-C ITEMS
# ============================================================

client = Client.open(API_ROOT)

search_kwargs = {
    "collections": [COLLECTION_ID],
    "bbox": BBOX,
    "datetime": DATETIME_RANGE,
    "limit": PAGE_LIMIT,
}

try:
    search = client.search(
        **search_kwargs,
        filter_lang="cql2-json",
        filter={
            "op": "like",
            "args": [
                {"property": "id"},
                ITEM_ID_LIKE,
            ],
        },
    )
    items = list(search.items())

except Exception as error:
    print(
        "Server-side LC-C filter failed; using local filtering:",
        error,
    )
    items = list(
        client.search(**search_kwargs).items()
    )

if not items:
    raise RuntimeError(
        "The STAC search returned no items. Check the collection, "
        "date interval and bounding box."
    )

print("STAC items returned:", len(items))
print("First item IDs:")
for item in items[:5]:
    print(" -", item.id)


STAC items returned: 2498
First item IDs:
 - fao-gismgr:ASIS:raster:mapsets:MVHI-D:items:ASIS-MVHI-D-2026-09-D2-GS2-LC-C
 - fao-gismgr:ASIS:raster:mapsets:MVHI-D:items:ASIS-MVHI-D-2026-09-D2-GS1-LC-C
 - fao-gismgr:ASIS:raster:mapsets:MVHI-D:items:ASIS-MVHI-D-2026-09-D1-GS2-LC-C
 - fao-gismgr:ASIS:raster:mapsets:MVHI-D:items:ASIS-MVHI-D-2026-09-D1-GS1-LC-C
 - fao-gismgr:ASIS:raster:mapsets:MVHI-D:items:ASIS-MVHI-D-2026-08-D2-GS2-LC-C


In [4]:
# ============================================================
# 4. LOAD AFGHANISTAN BOUNDARY
# ============================================================

ATTRIBUTE_NAME = "ISO3CD"
ATTRIBUTE_VALUE = "AFG"

if not BOUNDARY_FILE.exists():
    raise FileNotFoundError(
        f"Boundary file not found: {BOUNDARY_FILE}"
    )

geom = None
boundary_crs = None

with fiona.open(str(BOUNDARY_FILE)) as boundary_source:
    boundary_crs = (
        boundary_source.crs_wkt
        if boundary_source.crs_wkt
        else boundary_source.crs
    )

    if ATTRIBUTE_NAME not in boundary_source.schema["properties"]:
        raise KeyError(
            f"Field {ATTRIBUTE_NAME!r} was not found in "
            f"{BOUNDARY_FILE}. Available fields: "
            f"{list(boundary_source.schema['properties'])}"
        )

    for feature in boundary_source:
        feature_value = feature["properties"][ATTRIBUTE_NAME]

        if str(feature_value).strip().upper() == ATTRIBUTE_VALUE:
            geom = feature["geometry"]
            break

if geom is None:
    raise ValueError(
        f"Country {ATTRIBUTE_VALUE} was not found in "
        f"{BOUNDARY_FILE}"
    )

if not boundary_crs:
    raise ValueError(
        f"Boundary CRS is missing: {BOUNDARY_FILE}"
    )

print("Country boundary found:", ATTRIBUTE_VALUE)
print("Boundary CRS:", boundary_crs)


Country boundary found: AFG
Boundary CRS: GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]


In [5]:
# ============================================================
# 5. COMMON FUNCTIONS AND ITEM GROUPING
# ============================================================

# Current item format, after removal of the STAC namespace:
# ASIS-MVHI-D-2025-11-D1-GS2-LC-C
CURRENT_ITEM_PATTERN = re.compile(
    r"^ASIS-MVHI-D-"
    r"(?P<year>\d{4})-"
    r"(?P<phase>(?:0[1-9]|1[0-2])-D[123])-"
    r"(?P<season>GS[12])-"
    r"LC-C$"
)

# Compatibility with the former dot-separated item IDs.
LEGACY_ITEM_PATTERN = re.compile(
    r"^ASIS\.MVHI-D\."
    r"(?P<year>\d{4})-"
    r"(?P<phase>(?:0[1-9]|1[0-2])-D[123])\."
    r"(?P<season>GS[12])\."
    r"LC-C$"
)


def get_short_item_id(item_or_id):
    """Remove the optional '...:items:' STAC namespace."""

    raw_item_id = (
        item_or_id.id
        if hasattr(item_or_id, "id")
        else str(item_or_id)
    )
    return raw_item_id.strip().rsplit(":items:", 1)[-1]


def parse_item_id(item_or_id):
    """Parse current and legacy MVHI-D LC-C item IDs."""

    item_name = get_short_item_id(item_or_id)

    for pattern in (
        CURRENT_ITEM_PATTERN,
        LEGACY_ITEM_PATTERN,
    ):
        match_result = pattern.fullmatch(item_name)

        if match_result is not None:
            return {
                "item_name": item_name,
                "year": int(match_result.group("year")),
                "phase": match_result.group("phase"),
                "season": match_result.group("season"),
            }

    return None


def phase_sort_key(phase):
    month_text, dekad_text = phase.split("-")
    return (
        int(month_text),
        int(dekad_text.removeprefix("D")),
    )


def get_gcs_path(item):
    """Return bucket/object without the gs:// prefix."""

    assets = []

    if "data" in item.assets:
        assets.append(item.assets["data"])

    assets.extend(
        asset
        for asset_name, asset in item.assets.items()
        if asset_name != "data"
    )

    hrefs = []

    for asset in assets:
        alternate = asset.extra_fields.get("alternate", {})
        gs_entry = alternate.get("gs")

        if isinstance(gs_entry, dict) and gs_entry.get("href"):
            hrefs.append(gs_entry["href"])
        elif isinstance(gs_entry, str):
            hrefs.append(gs_entry)

        if asset.href:
            hrefs.append(asset.href)

    for href in hrefs:
        if href.startswith("gs://"):
            return href.removeprefix("gs://")

        parsed_href = urlparse(href)

        if parsed_href.netloc in {
            "storage.googleapis.com",
            "storage.cloud.google.com",
        }:
            return parsed_href.path.lstrip("/")

        if parsed_href.netloc.endswith(
            ".storage.googleapis.com"
        ):
            bucket = parsed_href.netloc.removesuffix(
                ".storage.googleapis.com"
            )
            return (
                f"{bucket}/"
                f"{parsed_href.path.lstrip('/')}"
            )

    raise ValueError(
        f"{item.id}: no usable GCS GeoTIFF asset found"
    )


def read_afghanistan_clip(path):
    """Clip one raster and mask NoData and values outside 0-1."""

    with rasterio.open(path) as src:
        if src.crs is None:
            raise ValueError(
                f"Raster CRS is missing: {path}"
            )

        raster_geom = transform_geom(
            boundary_crs,
            src.crs,
            geom,
        )

        clipped, output_transform = mask(
            src,
            [raster_geom],
            indexes=1,
            crop=True,
            filled=False,
            all_touched=False,
        )

        # Source NoData and pixels outside Afghanistan become NaN.
        data = clipped.astype("float32").filled(np.nan)
        profile = src.profile.copy()
        crs = src.crs

    valid = (
        np.isfinite(data)
        & (data >= VALID_MIN)
        & (data <= VALID_MAX)
    )
    data[~valid] = np.nan

    return data, valid, output_transform, profile, crs


# groups[(phase, season)][year] = item
groups = defaultdict(dict)
unmatched_item_ids = []

for item in items:
    parsed_item = parse_item_id(item)

    if parsed_item is None:
        unmatched_item_ids.append(item.id)
        continue

    key = (
        parsed_item["phase"],
        parsed_item["season"],
    )
    year = parsed_item["year"]

    if year in groups[key]:
        raise RuntimeError(
            f"Duplicate item for {key[0]} {key[1]} {year}: "
            f"{groups[key][year].id} and {item.id}"
        )

    groups[key][year] = item

groups = dict(groups)

if not groups:
    print("First returned item IDs:")
    for item in items[:5]:
        print(" -", repr(item.id))

    raise RuntimeError(
        "No recognized MVHI-D LC-C items were found. "
        "The item-ID examples above can be used to update the parser."
    )

available_phases = sorted(
    {phase for phase, _season in groups},
    key=phase_sort_key,
)

phases = (
    available_phases
    if PHASES_TO_RUN is None
    else PHASES_TO_RUN
)

matched_item_count = sum(
    len(year_to_item)
    for year_to_item in groups.values()
)

print("Matched MVHI-D LC-C items:", matched_item_count)
print("Unmatched items:", len(unmatched_item_ids))
print("Available phases:", available_phases)
print("Phases to process:", phases)


Matched MVHI-D LC-C items: 2498
Unmatched items: 0
Available phases: ['01-D1', '01-D2', '01-D3', '02-D1', '02-D2', '02-D3', '03-D1', '03-D2', '03-D3', '04-D1', '04-D2', '04-D3', '05-D1', '05-D2', '05-D3', '06-D1', '06-D2', '06-D3', '07-D1', '07-D2', '07-D3', '08-D1', '08-D2', '08-D3', '09-D1', '09-D2', '09-D3', '10-D1', '10-D2', '10-D3', '11-D1', '11-D2', '11-D3', '12-D1', '12-D2', '12-D3']
Phases to process: ['01-D1', '01-D2', '01-D3', '02-D1', '02-D2', '02-D3', '03-D1', '03-D2', '03-D3', '04-D1', '04-D2', '04-D3', '05-D1', '05-D2', '05-D3', '06-D1', '06-D2', '06-D3', '07-D1', '07-D2', '07-D3', '08-D1', '08-D2', '08-D3', '09-D1', '09-D2', '09-D3', '10-D1', '10-D2', '10-D3', '11-D1', '11-D2', '11-D3', '12-D1', '12-D2', '12-D3']


In [ ]:
# ============================================================
# 6. DOWNLOAD, CLIP AND CALCULATE MULTIYEAR MEANS
# ============================================================

gcs = gcsfs.GCSFileSystem(token=GCS_TOKEN)

# Check GCS read access before starting the long calculation.
sample_item = next(iter(next(iter(groups.values())).values()))
sample_path = get_gcs_path(sample_item)
gcs.info(sample_path)
print("GCS read access: OK")


def mean_output_path(phase, season, years):
    return OUTPUT_DIR / (
        f"ASIS_MVHI-D_AFG_mean_"
        f"{phase}_{season}_LC-C_"
        f"{years[0]}-{years[-1]}.tif"
    )


def calculate_mean(
    phase,
    season,
    year_to_item,
    phase_temp_dir,
):
    years_expected = sorted(year_to_item)
    output_file = mean_output_path(
        phase,
        season,
        years_expected,
    )

    if output_file.exists() and not OVERWRITE_MEANS:
        print(
            "  Already exists, skipped:",
            output_file.name,
        )
        return output_file, "skipped"

    pixel_sum = None
    pixel_count = None
    reference_shape = None
    output_transform = None
    output_profile = None
    output_crs = None

    years_used = []
    item_failures = []

    for item_number, year in enumerate(
        years_expected,
        start=1,
    ):
        item = year_to_item[year]
        item_name = get_short_item_id(item)
        local_file = phase_temp_dir / f"{item_name}.tif"

        print(
            f"  {season} "
            f"[{item_number:02d}/{len(years_expected):02d}] "
            f"{item_name}",
            flush=True,
        )

        try:
            gcs.get(
                get_gcs_path(item),
                str(local_file),
            )

            (
                data,
                valid,
                transform,
                profile,
                crs,
            ) = read_afghanistan_clip(local_file)

            if pixel_sum is None:
                reference_shape = data.shape
                output_transform = transform
                output_profile = profile
                output_crs = crs

                pixel_sum = np.zeros(
                    reference_shape,
                    dtype="float64",
                )
                pixel_count = np.zeros(
                    reference_shape,
                    dtype="uint16",
                )

            else:
                same_grid = (
                    data.shape == reference_shape
                    and transform.almost_equals(
                        output_transform
                    )
                    and crs == output_crs
                )

                if not same_grid:
                    raise ValueError(
                        f"{item_name}: raster grid differs "
                        "from other years"
                    )

            pixel_sum[valid] += data[valid]
            pixel_count[valid] += 1
            years_used.append(year)

        except Exception as error:
            item_failures.append(
                (year, item.id, str(error))
            )
            print("    FAILED:", error)

        finally:
            if local_file.exists():
                local_file.unlink()
            gc.collect()

    if (
        pixel_sum is None
        or pixel_count is None
        or output_profile is None
    ):
        raise RuntimeError(
            f"{phase} {season}: all annual items failed"
        )

    if item_failures and STRICT_ALL_YEARS:
        failed_years = ", ".join(
            str(failure[0])
            for failure in item_failures
        )
        raise RuntimeError(
            f"{phase} {season}: "
            f"{len(item_failures)} year(s) failed "
            f"({failed_years}); incomplete mean was not saved"
        )

    mean_array = np.full(
        reference_shape,
        OUTPUT_NODATA,
        dtype="float32",
    )

    # Pixel denominator contains only valid annual observations.
    np.divide(
        pixel_sum,
        pixel_count,
        out=mean_array,
        where=pixel_count > 0,
    )

    output_profile.update(
        driver="GTiff",
        height=mean_array.shape[0],
        width=mean_array.shape[1],
        transform=output_transform,
        crs=output_crs,
        count=1,
        dtype="float32",
        nodata=OUTPUT_NODATA,
        compress="DEFLATE",
        predictor=3,
        tiled=True,
        blockxsize=256,
        blockysize=256,
        BIGTIFF="IF_SAFER",
    )

    partial_output = output_file.with_suffix(
        ".partial.tif"
    )

    try:
        with rasterio.open(
            partial_output,
            "w",
            **output_profile,
        ) as dst:
            dst.write(mean_array, 1)
            dst.set_band_description(
                1,
                "MVHI-D multiyear pixel mean",
            )
            dst.update_tags(
                country="AFG",
                collection=COLLECTION_ID,
                phase=phase,
                growing_season=season,
                land_cover="LC-C",
                valid_range="0,1",
                output_nodata=str(OUTPUT_NODATA),
                years_used=",".join(
                    map(str, years_used)
                ),
                number_of_years_used=str(
                    len(years_used)
                ),
                mean_method=(
                    "pixel-wise arithmetic mean of valid "
                    "annual observations"
                ),
            )

        os.replace(partial_output, output_file)

    finally:
        if partial_output.exists():
            partial_output.unlink()

    print("  Saved:", output_file)
    return output_file, "written"


completed_outputs = []
skipped_outputs = []
mean_failures = []
mean_outputs = {}

for phase_number, phase in enumerate(phases, start=1):
    print(
        f"\n========== "
        f"{phase_number:02d}/{len(phases):02d} "
        f"{phase} "
        f"=========="
    )

    phase_temp_dir = Path(
        tempfile.mkdtemp(
            prefix=f"{phase}_",
            dir=TEMP_ROOT,
        )
    )

    try:
        for season in SEASONS_TO_RUN:
            year_to_item = groups.get(
                (phase, season),
                {},
            )

            if not year_to_item:
                print(f"  {season}: no items, skipped")
                continue

            years = sorted(year_to_item)
            print(
                f"  {season}: {len(years)} years, "
                f"{years[0]}-{years[-1]}"
            )

            try:
                output_file, status = calculate_mean(
                    phase=phase,
                    season=season,
                    year_to_item=year_to_item,
                    phase_temp_dir=phase_temp_dir,
                )
                mean_outputs[(phase, season)] = output_file

                if status == "written":
                    completed_outputs.append(output_file)
                else:
                    skipped_outputs.append(output_file)

            except Exception as error:
                mean_failures.append(
                    (phase, season, str(error))
                )
                print(
                    f"  FAILED: {phase} {season}: {error}"
                )

    finally:
        shutil.rmtree(
            phase_temp_dir,
            ignore_errors=True,
        )
        gc.collect()
        print(f"  Temporary data deleted: {phase}")

print("\n========== MEAN SUMMARY ==========")
print("New outputs:", len(completed_outputs))
print("Existing outputs skipped:", len(skipped_outputs))
print("Failures:", len(mean_failures))
print("Output directory:", OUTPUT_DIR)

if mean_failures:
    print("\nFailure details:")
    for phase, season, error in mean_failures:
        print(f"- {phase} {season}: {error}")


GCS read access: OK

========== 01/36 01-D1 ==========
  GS1: 35 years, 1992-2026
  GS1 [01/35] ASIS-MVHI-D-1992-01-D1-GS1-LC-C
  GS1 [02/35] ASIS-MVHI-D-1993-01-D1-GS1-LC-C
  GS1 [03/35] ASIS-MVHI-D-1994-01-D1-GS1-LC-C
  GS1 [04/35] ASIS-MVHI-D-1995-01-D1-GS1-LC-C
  GS1 [05/35] ASIS-MVHI-D-1996-01-D1-GS1-LC-C
  GS1 [06/35] ASIS-MVHI-D-1997-01-D1-GS1-LC-C
  GS1 [07/35] ASIS-MVHI-D-1998-01-D1-GS1-LC-C
  GS1 [08/35] ASIS-MVHI-D-1999-01-D1-GS1-LC-C
  GS1 [09/35] ASIS-MVHI-D-2000-01-D1-GS1-LC-C
  GS1 [10/35] ASIS-MVHI-D-2001-01-D1-GS1-LC-C
  GS1 [11/35] ASIS-MVHI-D-2002-01-D1-GS1-LC-C
  GS1 [12/35] ASIS-MVHI-D-2003-01-D1-GS1-LC-C


In [ ]:
# ============================================================
# 7. DISPLAY AND SAVE LATEST GS1 VS SAME-PHASE MEAN
# ============================================================

def select_latest_item(groups, season):
    candidates = []

    for (
        phase,
        group_season,
    ), year_to_item in groups.items():
        if group_season != season:
            continue

        month, dekad = phase_sort_key(phase)

        for year, item in year_to_item.items():
            candidates.append(
                ((year, month, dekad), item, phase)
            )

    if not candidates:
        raise RuntimeError(
            f"No MVHI-D {season} LC-C item was found"
        )

    (
        (latest_year, _month, _dekad),
        latest_item,
        latest_phase,
    ) = max(
        candidates,
        key=lambda record: record[0],
    )

    return latest_item, latest_year, latest_phase


def find_mean_file(phase, season, year_to_item):
    prefix = (
        f"ASIS_MVHI-D_AFG_mean_"
        f"{phase}_{season}_LC-C_"
    )
    mean_pattern = re.compile(
        re.escape(prefix)
        + r"(?P<first>\d{4})-(?P<last>\d{4})\.tif$"
    )

    if MEAN_FILE_OVERRIDE is not None:
        mean_file = Path(MEAN_FILE_OVERRIDE)

        if not mean_file.exists():
            raise FileNotFoundError(
                f"MEAN_FILE_OVERRIDE does not exist: "
                f"{mean_file}"
            )

        match_result = mean_pattern.fullmatch(
            mean_file.name
        )

        if match_result is None:
            raise ValueError(
                "MEAN_FILE_OVERRIDE does not match "
                f"{phase} {season} LC-C"
            )

        return (
            mean_file,
            (
                f"{match_result.group('first')}-"
                f"{match_result.group('last')}"
            ),
        )

    years = sorted(year_to_item)
    expected_file = mean_output_path(
        phase,
        season,
        years,
    )

    if expected_file.exists():
        return expected_file, f"{years[0]}-{years[-1]}"

    candidates = []

    for candidate in OUTPUT_DIR.glob(prefix + "*.tif"):
        match_result = mean_pattern.fullmatch(
            candidate.name
        )

        if candidate.is_file() and match_result is not None:
            first_year = int(
                match_result.group("first")
            )
            last_year = int(
                match_result.group("last")
            )
            candidates.append(
                (last_year, first_year, candidate)
            )

    if not candidates:
        raise FileNotFoundError(
            f"No mean GeoTIFF for {phase} {season} "
            f"was found in {OUTPUT_DIR}"
        )

    last_year, first_year, mean_file = max(candidates)
    print(
        "Expected mean was not found; using:",
        mean_file,
    )
    return mean_file, f"{first_year}-{last_year}"


def draw_boundary(axis, geometry):
    geometry_type = geometry["type"]

    if geometry_type == "Polygon":
        polygons = [geometry["coordinates"]]
    elif geometry_type == "MultiPolygon":
        polygons = geometry["coordinates"]
    elif geometry_type == "GeometryCollection":
        for part in geometry["geometries"]:
            draw_boundary(axis, part)
        return
    else:
        return

    for polygon in polygons:
        for ring in polygon:
            coordinates = np.asarray(ring)
            axis.plot(
                coordinates[:, 0],
                coordinates[:, 1],
                color="#555555",
                linewidth=0.7,
            )


latest_item, latest_year, latest_phase = select_latest_item(
    groups,
    LATEST_SEASON,
)

latest_year_to_item = groups[
    (latest_phase, LATEST_SEASON)
]

mean_file, mean_period = find_mean_file(
    latest_phase,
    LATEST_SEASON,
    latest_year_to_item,
)

print("Latest item:", latest_item.id)
print("Time phase:", latest_phase)
print("Mean file:", mean_file)

(
    mean_data,
    _mean_valid,
    mean_transform,
    _mean_profile,
    mean_crs,
) = read_afghanistan_clip(mean_file)

with tempfile.TemporaryDirectory(
    prefix=f"display_{LATEST_SEASON}_",
    dir=TEMP_ROOT,
) as temporary_directory:
    latest_name = get_short_item_id(latest_item)
    latest_file = (
        Path(temporary_directory) / f"{latest_name}.tif"
    )

    print("Downloading latest raster:", latest_name)
    gcs.get(
        get_gcs_path(latest_item),
        str(latest_file),
    )

    (
        latest_data,
        _latest_valid,
        latest_transform,
        _latest_profile,
        latest_crs,
    ) = read_afghanistan_clip(latest_file)

same_grid = (
    latest_data.shape == mean_data.shape
    and latest_transform.almost_equals(mean_transform)
    and latest_crs == mean_crs
)

if not same_grid:
    aligned_data = np.full(
        mean_data.shape,
        np.nan,
        dtype="float32",
    )

    reproject(
        source=latest_data,
        destination=aligned_data,
        src_transform=latest_transform,
        src_crs=latest_crs,
        src_nodata=np.nan,
        dst_transform=mean_transform,
        dst_crs=mean_crs,
        dst_nodata=np.nan,
        resampling=Resampling.nearest,
    )

    latest_data = aligned_data
    latest_transform = mean_transform
    latest_crs = mean_crs
    print("Latest raster aligned to the mean grid.")

plot_geom = transform_geom(
    boundary_crs,
    mean_crs,
    geom,
)

inside_afghanistan = geometry_mask(
    [plot_geom],
    out_shape=mean_data.shape,
    transform=mean_transform,
    invert=True,
    all_touched=False,
)

latest_data[~inside_afghanistan] = np.nan
mean_data[~inside_afghanistan] = np.nan

latest_data[
    ~np.isfinite(latest_data)
    | (latest_data < VALID_MIN)
    | (latest_data > VALID_MAX)
] = np.nan

mean_data[
    ~np.isfinite(mean_data)
    | (mean_data < VALID_MIN)
    | (mean_data > VALID_MAX)
] = np.nan

colour_map = plt.get_cmap("RdYlGn").copy()
colour_map.set_bad("#eeeeee")
normalizer = Normalize(
    vmin=VALID_MIN,
    vmax=VALID_MAX,
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(15, 7),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

titles = [
    (
        f"Latest {LATEST_SEASON} | "
        f"{latest_year}-{latest_phase}"
    ),
    (
        f"Multiyear mean {LATEST_SEASON} | "
        f"{latest_phase}\n{mean_period}"
    ),
]

for axis, data, title in zip(
    axes,
    (latest_data, mean_data),
    titles,
):
    show(
        np.ma.masked_invalid(data),
        transform=mean_transform,
        ax=axis,
        cmap=colour_map,
        norm=normalizer,
        interpolation="nearest",
    )
    draw_boundary(axis, plot_geom)
    axis.set_title(title)
    axis.set_xlabel(
        "Longitude" if mean_crs.is_geographic else "X"
    )
    axis.set_ylabel(
        "Latitude" if mean_crs.is_geographic else "Y"
    )

    if mean_crs.is_geographic:
        centre_latitude = sum(axis.get_ylim()) / 2
        axis.set_aspect(
            1 / np.cos(np.deg2rad(centre_latitude))
        )

    if not np.isfinite(data).any():
        axis.text(
            0.5,
            0.5,
            "No valid MVHI-D pixels (0-1)",
            transform=axis.transAxes,
            ha="center",
            va="center",
        )

figure.colorbar(
    axes[0].images[0],
    ax=axes,
    shrink=0.80,
    ticks=np.linspace(0, 1, 6),
    label="MVHI-D (0-1)",
)

figure.suptitle(
    "Afghanistan crop-growth monitoring | "
    f"MVHI-D | {LATEST_SEASON} | Cropland (LC-C)"
)

figure_path = OUTPUT_DIR / (
    f"MVHI_D_AFG_{LATEST_SEASON}_latest_"
    f"{latest_year}-{latest_phase}_vs_mean_"
    f"{mean_period}.png"
)

if OVERWRITE_FIGURE or not figure_path.exists():
    figure.savefig(
        figure_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
        facecolor="white",
    )
    print("Figure saved:", figure_path)
else:
    print(
        "Figure already exists, not overwritten:",
        figure_path,
    )

plt.show()
plt.close(figure)

print(
    "Grey pixels: source NoData, outside Afghanistan, "
    "or values outside the valid range 0-1."
)
print("Temporary latest source raster deleted.")


## Outputs

The `output` directory contains:

- one pixel-wise multiyear mean GeoTIFF for each available phase and
  growing season;
- a 300-dpi PNG comparing the latest GS1 MVHI-D image with the
  multiyear GS1 mean for the same dekadal phase.

NoData values, pixels outside Afghanistan, and MVHI-D values outside
the valid range 0–1 are excluded from the calculation and displayed in
grey.
